In [15]:
classes = ['冒充公检法及政府机关类', '冒充军警购物类诈骗', '冒充电商物流客服类', '冒充领导、熟人类', '无风险',
       '网络婚恋、交友类', '网黑案件', '虚假信用服务类', '虚假网络投资理财类', '虚假购物、服务类']
test_text = "您好，抖音上发现有人提供替考驾驶证服务，每科仅需1400元，微信号为wei1in12345，如有需要请联系，我们保证快速通过考试。"

import torch
if torch.cuda.is_available():
    device = torch.device("cuda")

In [16]:
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertForSequenceClassification
import torch

lb = LabelEncoder()
lb.classes_ = classes
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')

def encode_texts(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt',
    )

model = BertForSequenceClassification.from_pretrained('bert-base-chinese', num_labels=len(classes)).to(device)
model.load_state_dict(torch.load('final_model.pth'))
if torch.cuda.is_available():
    model = model.cuda()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_8946/2216896272.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We r

In [17]:
output = model(**encode_texts([test_text]).to(device))
output = output.logits
output = torch.softmax(output, dim=1)
output = output.cpu().detach().numpy()
output = output[0]
output = list(zip(lb.classes_, output))
output = sorted(output, key=lambda x: x[1], reverse=True)
output

[('虚假购物、服务类', 0.99719477),
 ('冒充军警购物类诈骗', 0.00060896657),
 ('虚假网络投资理财类', 0.0003540023),
 ('虚假信用服务类', 0.00034673887),
 ('冒充电商物流客服类', 0.00032761614),
 ('网黑案件', 0.0003237243),
 ('冒充领导、熟人类', 0.0002736996),
 ('无风险', 0.000227808),
 ('网络婚恋、交友类', 0.00019030586),
 ('冒充公检法及政府机关类', 0.00015232427)]